# BERT Text Length Experiment

This notebook investigates BERT's robustness at different text lengths by randomly dropping portions of text and measuring the impact on classification performance. We focus on Allam model training and test on multiple AI models.

In [1]:
import os
import shutil
import json
import random
import numpy as np

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"

In [2]:
import logging

loggers = [logging.getLogger(name) for name in logging.root.manager.loggerDict]
for logger in loggers:
    if "transformers" in logger.name.lower():
        logger.setLevel(logging.ERROR)

In [3]:
from models.data import ArabicSocialMediaDataModule

In [4]:
# Text dropping function
def drop_text(text, drop_percentage):
    if drop_percentage == 0.0:
        return text
        
    words = text.strip().split()
    if len(words) == 0:
        return text
    
    # Use consistent random seed for reproducibility
    rng = random.Random(hash(text) % (2**32))
    
    kept_words = []
    threshold = drop_percentage * 100  # Convert to 0-100 scale
    
    for word in words:
        # Generate random number between 0-100
        rand_num = int(rng.random() * 100)
        
        # Keep word if random number is greater than threshold
        if rand_num >= threshold:
            kept_words.append(word)
    
    # Ensure at least one word remains to avoid empty strings
    if len(kept_words) == 0 and len(words) > 0:
        random_index = rng.randint(0, len(words) - 1)
        kept_words = [words[random_index]]  # Keep a random word
        
    return ' '.join(kept_words)

In [5]:
from models.models import LitXLMRobertaModel
import pytorch_lightning as pl
from pytorch_lightning.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
)
from tabulate import tabulate

In [6]:
class TextLengthRobustnessExperiment:
    def __init__(self, train_model, max_epochs=100, results_file=None):
        self.max_epochs = max_epochs
        self.results_file = results_file
        self.train_model = train_model  # Focus only on Allam for training
        if self.results_file is None:
            self.results_file = f"notebooks/Arabic_experiments/ArabicSocialMediaDataset/text_lengths_experiments/{train_model}_text_length_results.json"
        self.checkpoints_path = "trained_detectors/Arabic/ArabicSocialMediaDataset/TextLengthExperiments/{drop_percentage}drop/checkpoints"
        self.test_models = ["allam", "jais-batched", "llama-batched", "openai"]
        self.drop_percentages = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 1.0]
        
        # Load existing results if available
        self.all_results = self._load_results()

    def _load_results(self):
        """Load existing results from JSON file."""
        if os.path.exists(self.results_file):
            try:
                with open(self.results_file, 'r') as f:
                    return json.load(f)
            except Exception as e:
                print(f"Warning: Could not load results from {self.results_file}")
        return {}
    
    def _save_results(self):
        """Save results to JSON file."""
        with open(self.results_file, 'w') as f:
            json.dump(self.all_results, f, indent=4, ensure_ascii=False)
        print(f"Results saved to {self.results_file}")

    def _get_callbacks(self, drop_percentage):
        early_stopping = EarlyStopping(
            monitor="val_loss",
            min_delta=0.0,
            patience=5,
            verbose=True,
            mode="min",
        )

        checkpoint = ModelCheckpoint(
            monitor="val_loss",
            dirpath=self.checkpoints_path.format(drop_percentage=f"{drop_percentage:.0%}"),
            filename="best-checkpoint",
            save_top_k=1,
            mode="min",
        )

        return [early_stopping, checkpoint]

    def _test_on_model(self, trainer, model, test_model, drop_percentage):
        # Load the best checkpoint before testing
        checkpoint_path = self.checkpoints_path.format(drop_percentage=f"{drop_percentage:.0%}")
        checkpoint_path += "/best-checkpoint.ckpt"
        model = LitXLMRobertaModel.load_from_checkpoint(checkpoint_path)
        model.eval()
        
        # Create test datamodule with the same text processor
        text_processor = lambda text: drop_text(text, drop_percentage)  # noqa: E731
        test_datamodule = ArabicSocialMediaDataModule(
            models=[test_model], 
            text_processors=text_processor
        )
        test_datamodule.setup()

        results = trainer.test(model, test_datamodule.test_dataloader())[0]
        return {
            "accuracy": results["test_acc"],
            "precision": results["test_precision"],
            "recall": results["test_recall"],
            "f1": results["test_f1"],
            "loss": results["test_loss"],
        }

    def run_single_experiment(self, drop_percentage, run_number):
        """Run a single experiment for a specific drop percentage and run number."""
        print(f"\n{'='*60}")
        print(f"Drop Percentage: {drop_percentage:.0%}, Run: {run_number + 1}/10")
        print(f"{'='*60}")
        
        # Create text processor function
        text_processor = lambda text: drop_text(text, drop_percentage) # noqa: E731
        
        # Initialize model and data module
        model = LitXLMRobertaModel()
        train_datamodule = ArabicSocialMediaDataModule(
            models=[self.train_model], 
            text_processors=text_processor
        )
        
        # Initialize trainer
        trainer = pl.Trainer(
            devices=1,
            max_epochs=self.max_epochs,
            accelerator="auto",
            val_check_interval=0.25,
            check_val_every_n_epoch=1,
            callbacks=self._get_callbacks(drop_percentage),
        )

        # Train the model
        print(f"Training on {self.train_model} data with {drop_percentage:.0%} word drop...")
        model.train()
        trainer.fit(model, train_datamodule)

        # Test on all specified models
        results = {}
        for test_model in self.test_models:
            model.eval()
            print(f"Testing on {test_model} data...")
            results[test_model] = self._test_on_model(
                trainer, model, test_model, drop_percentage
            )

        # Display results for this run
        self._display_single_run_results(drop_percentage, run_number, results)

        # Clean up checkpoints to save space
        checkpoint_dir = self.checkpoints_path.format(drop_percentage=f"{drop_percentage:.0%}")
        print(f"Cleaning up checkpoint directory: {checkpoint_dir}")
        shutil.rmtree(checkpoint_dir, ignore_errors=True)

        return results

    def _display_single_run_results(self, drop_percentage, run_number, results):
        """Display results for a single run."""
        print(f"\nResults for {drop_percentage:.0%} drop, run {run_number + 1}:")
        print("-" * 80)
        print(f"{'Test Model':<15} {'Accuracy':<10} {'Precision':<10} {'Recall':<10} {'F1':<10} {'Loss':<10}")
        print("-" * 80)
        for test_model, metrics in results.items():
            print(
                f"{test_model:<15}"
                f"{metrics['accuracy']:<10.4f}"
                f"{metrics['precision']:<10.4f}"
                f"{metrics['recall']:<10.4f}"
                f"{metrics['f1']:<10.4f}"
                f"{metrics['loss']:<10.4f}"
            )
        print("-" * 80)

    def run_all_experiments(self, num_runs=10):
        """Run the complete experiment with multiple runs for each drop percentage."""
        for drop_percentage in self.drop_percentages:
            drop_key = f"{drop_percentage:.0%}"
            
            if drop_key not in self.all_results:
                self.all_results[drop_key] = []
            
            # Continue from where we left off
            completed_runs = len(self.all_results[drop_key])
            
            for run_number in range(completed_runs, num_runs):
                print(f"\nStarting experiment for {drop_percentage:.0%} drop, run {run_number + 1}/{num_runs}")
                try:
                    results = self.run_single_experiment(drop_percentage, run_number)
                    self.all_results[drop_key].append(results)
                    
                    # Save results after each run
                    self._save_results()
                    
                except Exception as e:
                    print(f"Error in run {run_number + 1} for {drop_percentage:.0%} drop: {str(e)}")
                    # Save what we have so far
                    self._save_results()
                    raise

    def calculate_statistics(self):
        """Calculate mean and std for all metrics across runs."""
        stats = {}
        
        for drop_key, runs in self.all_results.items():
            if not runs:
                continue
                
            stats[drop_key] = {}
            
            for test_model in self.test_models:
                # Collect metrics across all runs
                metrics_data = {
                    'accuracy': [run[test_model]['accuracy'] for run in runs if test_model in run],
                    'precision': [run[test_model]['precision'] for run in runs if test_model in run],
                    'recall': [run[test_model]['recall'] for run in runs if test_model in run],
                    'f1': [run[test_model]['f1'] for run in runs if test_model in run],
                    'loss': [run[test_model]['loss'] for run in runs if test_model in run],
                }
                
                # Calculate mean and std
                stats[drop_key][test_model] = {}
                for metric, values in metrics_data.items():
                    if values:
                        stats[drop_key][test_model][f'{metric}_mean'] = np.mean(values)
                        stats[drop_key][test_model][f'{metric}_std'] = np.std(values, ddof=1)  # Sample std
                        
        return stats

    def print_summary_table(self):
        """Print a summary table with mean ± std for each metric."""
        stats = self.calculate_statistics()
        
        for metric in ['accuracy', 'precision', 'recall', 'f1', 'loss']:
            print(f"\n{metric.upper()} Results (Mean ± Std):")
            print("=" * 80)
            
            # Prepare table data
            headers = ['Drop %'] + self.test_models
            table_data = []
            
            for drop_key in sorted(stats.keys(), key=lambda x: float(x.rstrip('%'))):
                row = [drop_key]
                for test_model in self.test_models:
                    if test_model in stats[drop_key]:
                        mean_val = stats[drop_key][test_model].get(f'{metric}_mean', 0)
                        std_val = stats[drop_key][test_model].get(f'{metric}_std', 0)
                        row.append(f"{mean_val:.4f} ± {std_val:.4f}")
                    else:
                        row.append("N/A")
                table_data.append(row)
            
            print(tabulate(table_data, headers=headers, tablefmt='grid'))

In [7]:
# Initialize experiment
train_model = 'allam'
experiment = TextLengthRobustnessExperiment(
    max_epochs=100,
    train_model=train_model,
    results_file=f"notebooks/Arabic_experiments/ArabicSocialMediaDataset/text_lengths_experiments/{train_model}_text_length_results.json"
)

print("Starting BERT Text Length Experiments")
print(f"Drop percentages: {[f'{p:.0%}' for p in experiment.drop_percentages]}")
print(f"Training model: {experiment.train_model}")
print(f"Test models: {experiment.test_models}")
print("Number of runs per drop percentage: 10")

Starting BERT Text Length Experiments
Drop percentages: ['0%', '5%', '10%', '15%', '20%', '25%', '30%', '35%', '40%', '45%', '50%', '55%', '60%', '65%', '70%', '75%', '80%', '85%', '90%', '95%', '100%']
Training model: allam
Test models: ['allam', 'jais-batched', 'llama-batched', 'openai']
Number of runs per drop percentage: 10


In [8]:
# Run the experiments
experiment.run_all_experiments(num_runs=10)


Starting experiment for 35% drop, run 4/10

Drop Percentage: 35%, Run: 4/10


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


/raid_storage/SLURM/home/slurm_majedalshaibani/Projects/arabic-text-detection/.venv/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python3 /raid_storage/SLURM/home/slurm_majedalshaibani/Proj ...
GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


HPU available: False, using: 0 HPUs


...

Validation: |          | 0/? [00:00<?, ?it/s]

Metric val_loss improved by 0.032 >= min_delta = 0.0. New best score: 0.305


Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Metric val_loss improved by 0.002 >= min_delta = 0.0. New best score: 0.304


In [ ]:
# Print summary tables
experiment.print_summary_table()

In [ ]:
# Show final results file location
print(f"\nFinal results saved to: {experiment.results_file}")
print(f"Results file size: {os.path.getsize(experiment.results_file) / 1024:.2f} KB")